## Google Colab / GPU setup

In [ ]:
# Detect Colab if present
try:
    from google.colab import drive
    COLAB = True
    print('Note: using Google Colab')
except:
    print('Note: not using Google Colab')
    COLAB = False

# Use GPU or MPS (Apple) if available
import torch
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## FTIR Images Dataset

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

# Dataset class to retrieve malignant and non-malignant FTIR images
class FTIRDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.data = []
        self.load_data()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        img = Image.open(img_path)
        return img, label
    
    def load_data(self):
        for label, sub_dir in enumerate(['non_malignant', 'malignant']):
            full_path = os.path.join(self.root_dir, sub_dir)
            images = [(os.path.join(full_path, img), label) for img in os.listdir(full_path) if img.endswith('.tif')]
            self.data.extend(images)
            

# Dataset: 78 FTIR core images total (65 malignant, 13 non-malignant)
full_dataset = FTIRDataset(root_dir='../dataset/ftir_core_all_images')
print(f'Dataset samples: {len(full_dataset)}')

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms

IMAGE_WIDTH = 224
IMAGE_HEIGHT = 224
BATCH_SIZE = 8

# Augment training images (due to small set size)
def get_train_transform():
    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_WIDTH, IMAGE_HEIGHT)),
        transforms.Grayscale(num_output_channels=3),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(90, fill=(0, 0, 0)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return train_transform


# Test transform (no augmentation)
def get_test_transform():
    test_transform = transforms.Compose([
        transforms.Resize((IMAGE_WIDTH, IMAGE_HEIGHT)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return test_transform


# Applies augmentations to dataset (necessary to ensure training subset is augmented but not validation subset in cross-validation later)
class WrapperDataset(Dataset):
    def __init__(self, base_dataset, transform=None):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        
        if self.transform:
            img = self.transform(img)

        return img, label


augmented_dataset = WrapperDataset(base_dataset=full_dataset, transform=get_train_transform())
dataloader = DataLoader(augmented_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Check first batch as an example
for images, labels in dataloader:
    print(f"Batch shape: {images.shape}, Batch labels: {labels}")
    break

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random

# Display random augmented images from the dataset 
def display_images(dataset, num_images=5, title=''):
    fig, axes = plt.subplots(1, num_images, figsize=(15, 5))
    fig.suptitle(title, fontsize=16)
    fig.subplots_adjust(top=0.95)
    indices = random.sample(range(len(dataset)), num_images) 

    for i, ax in enumerate(axes):
        img, label = dataset[indices[i]]
        img = img.permute(1, 2, 0).numpy()
        img = img[:, :, 0] 

        ax.imshow(img, cmap='gray')
        ax.axis('off')
        ax.set_title(f'Label: {label}')

    legend_text = '0: non-malignant, 1: malignant'
    fig.text(0.5, 0.85, legend_text, ha='center', fontsize=12, bbox=dict(facecolor='white', alpha=0.5))
    plt.show()
    

display_images(augmented_dataset, title='Example Images from Augmented Dataset')

## Model Initialisation

A pretrained Resnet-18 model was used (with ImageNet weights) to achieve better results via transfer learning, due to the small dataset size (78 images total). 

Cross-entropy loss function was used with class weights set to [0.9, 0.1] for non-malignant class(0) and malignant class (1) respectively, to deal with the class imbalance at the core level (65 malignant images and 13 non-malignant images).

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torchvision import models
from torchvision.models import ResNet18_Weights

def init_model():
    model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2) # binary classification
    
    for name, param in model.named_parameters():
        if 'fc' in name: # unfreeze final classification layer
            param.requires_grad = True
        else:
            param.requires_grad = False
    
    model.to(device)
    return model

def init_optimiser(model, learning_rate=3e-4):
    optimiser = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    return optimiser

def init_loss_fn():
    weights = torch.tensor([0.85, 0.15]).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    return loss_fn


model = init_model()
#summary(model, input_size=(8, 3, 224, 224))

## Training (Cross-validation)

The CNN model was trained on the full FTIR images dataset using 5-fold stratified cross validation. For each fold, the dataset was divided into a training subset and an independent validation subset.

In [ ]:
# Split dataset into training subset (augmented) and validation subset
def setup_data_loaders(train_id, val_id, dataset):
    train_transform = get_train_transform()
    val_transform = get_test_transform()
    
    train_subset = WrapperDataset(Subset(dataset, train_id), transform=train_transform)
    val_subset = WrapperDataset(Subset(dataset, val_id), transform=val_transform)
    
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

def train(train_loader, model, optimiser, loss_fn):
    model.train()
    epoch_loss = 0
    preds = []
    targets = []

    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        # Forward pass
        y_batch_pred = model(x_batch)
        loss = loss_fn(y_batch_pred, y_batch)

        # Backward pass and optimisation (adam step)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        epoch_loss += loss.item()

        _, batch_pred = torch.max(y_batch_pred, 1)
        preds.extend(batch_pred.cpu().numpy())
        targets.extend(y_batch.cpu().numpy())
        

    avg_loss = epoch_loss / len(train_loader)
    accuracy = accuracy_score(targets, preds)
    return avg_loss, accuracy


def validate(val_loader, model, loss_fn):
    model.eval()
    total_loss = 0
    val_preds = []
    val_probs = []
    val_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

            val_labels.extend(labels.cpu().numpy())

            _, batch_pred = torch.max(outputs, 1)
            val_preds.extend(batch_pred.cpu().numpy())
            
            probabilities = F.softmax(outputs, dim=1)
            val_probs.extend(probabilities[:, 1].cpu().numpy())

    avg_loss = total_loss / len(val_loader)
    accuracy = accuracy_score(val_labels, val_preds)
    return avg_loss, accuracy, val_probs, val_labels

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
from tqdm.auto import tqdm

num_epochs = 50
num_folds = 5
kf = StratifiedKFold(num_folds, shuffle=True)
labels = [label for _, label in full_dataset.data]

all_train_losses = []
all_val_losses = []
fold_val_probs = []
fold_val_labels = []
fold_aucs = []

for fold, (train_id, val_id) in enumerate(kf.split(np.zeros(len(labels)), labels)):
    train_loader, val_loader = setup_data_loaders(train_id, val_id, full_dataset)
    model = init_model()
    optimiser = init_optimiser(model)
    loss_fn = init_loss_fn()

    fold_train_losses = []
    fold_val_losses = []

    with tqdm(total=num_epochs, desc=f'Fold {fold + 1}/{num_folds}', position=0, leave=True) as pbar:
        for epoch in range(num_epochs):
            train_loss, train_acc = train(train_loader, model, optimiser, loss_fn)
            val_loss, val_acc, val_probs, val_labels = validate(val_loader, model, loss_fn)
            
            fold_train_losses.append(train_loss)
            fold_val_losses.append(val_loss)
            fold_val_probs.extend(val_probs)
            fold_val_labels.extend(val_labels)

            roc_auc = roc_auc_score(val_labels, val_probs)
            fold_aucs.append(roc_auc)
            
            print(f'Epoch [{epoch + 1}/{num_epochs}]\nTrain Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}' \
                  f'\nTrain Acc: {train_acc * 100:.2f}%, Val Acc: {val_acc * 100:.2f}%\n')
            pbar.update(1)

    all_train_losses.append(fold_train_losses)
    all_val_losses.append(fold_val_losses)

## Results

In [ ]:
# Calculate average train and validation loss per epoch across all folds
avg_train_losses = np.mean(all_train_losses, axis=0)
avg_val_losses = np.mean(all_val_losses, axis=0)

# Plot train and validation loss
plt.figure(figsize=(10, 6))
plt.plot(avg_train_losses, label='Average Training Loss')
plt.plot(avg_val_losses, label='Average Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Average Training and Validation Loss per Epoch Across All Folds')
plt.legend()
plt.show()

In [ ]:
# Plot ROC curve for given probabilities and true labels
def plot_roc(preds, labels):
    fpr, tpr, _ = roc_curve(labels, preds)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc="lower right")
    plt.show()

plot_roc(fold_val_probs, fold_val_labels)
average_roc_auc = np.mean(fold_aucs)
print(f'Average ROC AUC across all folds: {average_roc_auc:.2f}')